In [1]:
%pip install opencv-python pandas tqdm -q

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121
Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install --upgrade ipywidgets widgetsnbextension jupyterlab_widgets

Note: you may need to restart the kernel to use updated packages.


In [4]:
import os, cv2, numpy as np, pandas as pd
from tqdm.notebook import tqdm
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

In [5]:
class SegDataset(Dataset):
    def __init__(self, img_dir, mask_dir=None, transform=None):
        self.img_dir, self.mask_dir = img_dir, mask_dir
        
        # Check if directories exist
        if not os.path.exists(img_dir):
            raise ValueError(f"Image directory does not exist: {img_dir}")
        if mask_dir and not os.path.exists(mask_dir):
            raise ValueError(f"Mask directory does not exist: {mask_dir}")
        
        # Get list of valid image files
        self.fnames = sorted([f for f in os.listdir(img_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
        if len(self.fnames) == 0:
            raise ValueError(f"No image files found in {img_dir}")
        
        self.transform = transform

    def __len__(self):
        return len(self.fnames)

    def __getitem__(self, idx):
        fname = self.fnames[idx]
        img_path = os.path.join(self.img_dir, fname)
        
        # Load image with error handling
        img = cv2.imread(img_path)
        if img is None:
            raise ValueError(f"Failed to load image: {img_path}")
        
        img = img[:, :, ::-1]  # BGR to RGB
        img = cv2.resize(img, (256, 256))
        img = self.transform(img) if self.transform else transforms.ToTensor()(img)

        if self.mask_dir:
            mask_path = os.path.join(self.mask_dir, fname)
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            if mask is None:
                raise ValueError(f"Failed to load mask: {mask_path}")
            
            mask = cv2.resize(mask, (256, 256), interpolation=cv2.INTER_NEAREST)
            return img, torch.from_numpy(mask).long()
        return img, fname

In [6]:
def rle_encode(mask):
    pixels = mask.flatten(order="F")
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[:-1:2]
    return " ".join(str(x) for x in runs)

In [7]:
def get_model(num_classes=16):
    model = models.segmentation.deeplabv3_resnet50(pretrained=True)
    model.classifier[4] = nn.Conv2d(256, num_classes, kernel_size=1)
    return model

In [8]:
def train_model(model, train_loader, device, epochs=10):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for imgs, masks in tqdm(train_loader):
            imgs, masks = imgs.to(device), masks.to(device)
            optimizer.zero_grad()
            out = model(imgs)["out"]
            loss = criterion(out, masks)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1}, loss={total_loss/len(train_loader):.4f}")
    return model

In [9]:
def inference_and_submit(model, test_loader, device, out_csv="submission.csv"):
    model.eval()
    records = []
    with torch.no_grad():
        for imgs, fnames in tqdm(test_loader):
            imgs = imgs.to(device)
            preds = model(imgs)["out"].argmax(1).cpu().numpy()
            for pred, fname in zip(preds, fnames):
                row = {"img": fname}
                for class_id in range(16):
                    class_mask = (pred == class_id).astype(np.uint8)
                    row[f"class_{class_id}"] = (
                        "none" if class_mask.sum() == 0 else rle_encode(class_mask)
                    )
                records.append(row)
    pd.DataFrame(records).to_csv(out_csv, index=False)
    print(f"Saved {out_csv}")

In [ ]:
# 設定路徑
train_img_dir = "./data/train/imgs"  # Changed from "./data/train"
train_mask_dir = "./data/train/masks"  # Changed from "./data/train_masks"
test_img_dir = "./data/test/imgs"  # Changed from "./data/test"

transform = transforms.Compose(
  [
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
  ]
)

# DataLoader
train_set = SegDataset(train_img_dir, train_mask_dir, transform)
train_loader = DataLoader(train_set, batch_size=16, shuffle=True)
test_set = SegDataset(test_img_dir, transform=transform)
test_loader = DataLoader(test_set, batch_size=16, shuffle=False)

# Model
device = "cuda" if torch.cuda.is_available() else "cpu"
model = get_model().to(device)

# 訓練
model = train_model(model, train_loader, device, epochs=5)

# 推論 + 產生 submission.csv
inference_and_submit(model, test_loader, device)

  0%|          | 0/250 [00:00<?, ?it/s]

Epoch 1, loss=1.2057


  0%|          | 0/250 [00:00<?, ?it/s]

Epoch 2, loss=0.7089


  0%|          | 0/250 [00:00<?, ?it/s]

Epoch 3, loss=0.5658


  0%|          | 0/250 [00:00<?, ?it/s]

In [ ]:
!kaggle competitions submit -c 2025-ncku-ee-ml-16-classes-segmentation -f submission.csv -m "baseline deeplabv3"